# Philosophy Psychology Wiki v1 Builder

Build `philosophy_psychology_wiki_v1`, the Wikipedia corpus for **Philosophy and Psychology (Great thinkers and the human psyche)**.

Design:

- Discovery starts from curated Wikipedia categories and manual seed pages.
- Pages are resolved to Wikidata QIDs through the Wikipedia API.
- Article text is fetched from `DragonLLM/Clean-Wikipedia-English-Articles` by QID.
- The dataset is saved under `Datasets/philosophy_psychology_wiki_v1`.
- The FAISS index is saved under `Indexes/philosophy_psychology_wiki_v1`.
- Index chunks use 512 tokens with 128 overlap, matching the existing wiki builders.

Coverage target:

- great thinkers, philosophers, psychologists, psychoanalysts
- philosophical schools, concepts, theories and branches
- political/social thought and ideology: Marxism, neoliberalism, socialism, liberalism
- economic ideologies: laissez-faire, free market, scientific socialism
- human psyche: psychology, psychoanalysis, cognition, behaviorism
- gender, identity, critical theory: Judith Butler, Derrida, queer theory, feminist philosophy
- applied ethics / society: CSR, business ethics, stakeholder theory, ESG

In [ ]:
# @title Mount Google Drive (Colab only)

# Run this cell only in Colab.
from google.colab import drive
drive.mount('/content/drive')

## 0. Setup

If dependencies are missing in Colab, uncomment the install cell. Locally, the repository `requirements.txt` already lists the packages used here.

In [ ]:
!pip -q install datasets pandas tqdm langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken openai

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import json
import math
import os
import re
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from tqdm.auto import tqdm

PROJECT_ROOT_OVERRIDE = None
PROJECT_MARKERS = [Path('requirements.txt'), Path('millionaire_client')]
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/NLP'),
    Path('/content/drive/MyDrive/Colab Notebooks/NLP'),
    Path('/gdrive/MyDrive/NLP'),
    Path('/gdrive/MyDrive/Colab Notebooks/NLP'),
]

def looks_like_project_root(path):
    return any((path / marker).exists() for marker in PROJECT_MARKERS)

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((candidate for candidate in PROJECT_ROOT_CANDIDATES if looks_like_project_root(candidate)), Path.cwd())

os.chdir(PROJECT_ROOT)

DATASETS_DIR = PROJECT_ROOT / 'Datasets'
INDEXES_DIR = PROJECT_ROOT / 'Indexes'
LOGS_DIR = PROJECT_ROOT / 'logs'
CACHE_DIR = LOGS_DIR / '.cache'

for directory in [DATASETS_DIR, INDEXES_DIR, LOGS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_LABEL = 'philosophy_psychology_wiki_v1'
SOURCE_WIKI_DATASET = 'DragonLLM/Clean-Wikipedia-English-Articles'
OUTPUT_DATASET_DIR = DATASETS_DIR / RUN_LABEL
OUTPUT_INDEX_DIR = INDEXES_DIR / RUN_LABEL

CANDIDATE_QIDS_CSV = LOGS_DIR / f'{RUN_LABEL}_candidate_qids.csv'
SELECTED_QIDS_CSV = LOGS_DIR / f'{RUN_LABEL}_selected_qids.csv'
FETCH_REPORT_CSV = LOGS_DIR / f'{RUN_LABEL}_fetch_report.csv'
ADDED_ARTICLES_CSV = LOGS_DIR / f'{RUN_LABEL}_added_articles.csv'
SKIPPED_ARTICLES_CSV = LOGS_DIR / f'{RUN_LABEL}_skipped_articles.csv'
BUILD_REPORT_JSON = LOGS_DIR / f'{RUN_LABEL}_build_report.json'
INDEX_REPORT_JSON = LOGS_DIR / f'{RUN_LABEL}_index_report.json'
DISCOVERY_CACHE_JSON = CACHE_DIR / f'{RUN_LABEL}_wikipedia_category_cache.json'
RESOLUTION_CACHE_JSON = CACHE_DIR / f'{RUN_LABEL}_title_resolution_cache.json'
MANUAL_SEED_QIDS_CSV = LOGS_DIR / f'{RUN_LABEL}_manual_seed_qids.csv'

RUN_WIKIPEDIA_DISCOVERY = True
RUN_FETCH_AND_BUILD_DATASET = True
RUN_INDEX_BUILD = True

OVERWRITE_OUTPUT_DATASET = False
OVERWRITE_INDEX = False
MAX_SOURCE_ROWS_TO_SCAN = None
SOURCE_SCAN_PROGRESS_EVERY = 250_000

MIN_ARTICLE_CHARS = 900
CATEGORY_CANDIDATE_MULTIPLIER = 4
MAX_CATEGORY_DEPTH_DEFAULT = 1
USER_AGENT = 'philosophy-psychology-wiki-v1-builder/1.0 (student NLP project; Wikipedia categories + DragonLLM QID filtering)'
WIKIPEDIA_LICENSE = 'cc-by-sa-4.0'
WIKI_SOURCE_TYPE = 'wikipedia_article'
TAXONOMY_SOURCE = 'wikipedia_category_qid_v1'

print('project root:', PROJECT_ROOT)
print('output dataset:', OUTPUT_DATASET_DIR)
print('output index:', OUTPUT_INDEX_DIR)
print('candidate qids:', CANDIDATE_QIDS_CSV)
print('selected qids:', SELECTED_QIDS_CSV)

## 1. Taxonomy, Categories, Manual Seeds

This category is intentionally broader than pure philosophy. The mock questions require political/economic thought, social theory, psychology and applied ethics.

In [ ]:
SUBJECT_QUOTAS = {
    'core_philosophy': 650,
    'political_social_thought': 560,
    'economic_ideologies': 260,
    'psychology_psyche': 470,
    'gender_identity_critical_theory': 280,
    'ethics_society_csr': 220,
}

SUBJECT_PRIORITY = [
    'core_philosophy',
    'political_social_thought',
    'psychology_psyche',
    'economic_ideologies',
    'gender_identity_critical_theory',
    'ethics_society_csr',
]

CATEGORY_SPECS = {
    'core_philosophy': [
        {'category': 'Philosophy', 'max_depth': 1, 'weight': 75},
        {'category': 'Philosophers', 'max_depth': 1, 'weight': 85},
        {'category': 'Philosophical concepts', 'max_depth': 1, 'weight': 95},
        {'category': 'Philosophical theories', 'max_depth': 1, 'weight': 90},
        {'category': 'Philosophical schools and traditions', 'max_depth': 1, 'weight': 95},
        {'category': 'Branches of philosophy', 'max_depth': 1, 'weight': 80},
        {'category': 'Epistemology', 'max_depth': 1, 'weight': 80},
        {'category': 'Metaphysics', 'max_depth': 1, 'weight': 80},
        {'category': 'Philosophy of mind', 'max_depth': 1, 'weight': 90},
        {'category': 'Ethics', 'max_depth': 1, 'weight': 75},
    ],
    'political_social_thought': [
        {'category': 'Political philosophy', 'max_depth': 1, 'weight': 95},
        {'category': 'Social philosophy', 'max_depth': 1, 'weight': 90},
        {'category': 'Political theories', 'max_depth': 1, 'weight': 90},
        {'category': 'Political ideologies', 'max_depth': 1, 'weight': 95},
        {'category': 'Social theories', 'max_depth': 1, 'weight': 90},
        {'category': 'Marxism', 'max_depth': 1, 'weight': 100},
        {'category': 'Socialism', 'max_depth': 1, 'weight': 80},
        {'category': 'Liberalism', 'max_depth': 1, 'weight': 75},
        {'category': 'Critical theory', 'max_depth': 1, 'weight': 90},
        {'category': 'Post-structuralism', 'max_depth': 1, 'weight': 85},
    ],
    'economic_ideologies': [
        {'category': 'Economic ideologies', 'max_depth': 1, 'weight': 100},
        {'category': 'History of economic thought', 'max_depth': 1, 'weight': 85},
        {'category': 'Capitalism', 'max_depth': 1, 'weight': 80},
        {'category': 'Economic liberalism', 'max_depth': 1, 'weight': 90},
        {'category': 'Free market', 'max_depth': 1, 'weight': 90},
        {'category': 'Political economy', 'max_depth': 1, 'weight': 80},
    ],
    'psychology_psyche': [
        {'category': 'Psychology', 'max_depth': 1, 'weight': 90},
        {'category': 'Psychologists', 'max_depth': 1, 'weight': 85},
        {'category': 'Psychological concepts', 'max_depth': 1, 'weight': 100},
        {'category': 'Psychological theories', 'max_depth': 1, 'weight': 95},
        {'category': 'Psychoanalysis', 'max_depth': 1, 'weight': 95},
        {'category': 'Cognitive science', 'max_depth': 1, 'weight': 80},
        {'category': 'Behaviorism', 'max_depth': 1, 'weight': 80},
        {'category': 'Human behavior', 'max_depth': 1, 'weight': 75},
    ],
    'gender_identity_critical_theory': [
        {'category': 'Feminist philosophy', 'max_depth': 1, 'weight': 95},
        {'category': 'Feminist theory', 'max_depth': 1, 'weight': 95},
        {'category': 'Queer theory', 'max_depth': 1, 'weight': 100},
        {'category': 'Gender studies', 'max_depth': 1, 'weight': 80},
        {'category': 'Postmodern philosophy', 'max_depth': 1, 'weight': 80},
        {'category': 'Continental philosophy', 'max_depth': 1, 'weight': 75},
    ],
    'ethics_society_csr': [
        {'category': 'Business ethics', 'max_depth': 1, 'weight': 100},
        {'category': 'Corporate social responsibility', 'max_depth': 1, 'weight': 100},
        {'category': 'Applied ethics', 'max_depth': 1, 'weight': 90},
        {'category': 'Environmental social science', 'max_depth': 1, 'weight': 70},
        {'category': 'Corporate governance', 'max_depth': 1, 'weight': 75},
        {'category': 'Sustainability', 'max_depth': 1, 'weight': 65},
    ],
}

MANUAL_SEED_TITLES = {
    'core_philosophy': [
        'Philosophy', 'Socrates', 'Plato', 'Aristotle', 'Rene Descartes', 'John Locke', 'David Hume',
        'Immanuel Kant', 'Georg Wilhelm Friedrich Hegel', 'Arthur Schopenhauer', 'Friedrich Nietzsche',
        'Soren Kierkegaard', 'Jean-Paul Sartre', 'Simone de Beauvoir', 'Michel Foucault', 'Jacques Derrida',
        'Ludwig Wittgenstein', 'Bertrand Russell', 'Gottlob Frege', 'Baruch Spinoza', 'Thomas Hobbes',
        'Jean-Jacques Rousseau', 'John Stuart Mill', 'Utilitarianism', 'Deontology', 'Virtue ethics',
        'Existentialism', 'Phenomenology (philosophy)', 'Hermeneutics', 'Analytic philosophy',
        'Continental philosophy', 'Empiricism', 'Rationalism', 'Idealism', 'Materialism', 'Dualism',
        'Monism', 'Metaphysics', 'Epistemology', 'Ontology', 'Aesthetics', 'Logic', 'Philosophy of mind',
        'Free will', 'Determinism', 'Consciousness', 'Personal identity', 'Subjective idealism',
        'Immaterialism', 'George Berkeley', 'Philosophy of perception', 'Allegory of the cave',
    ],
    'political_social_thought': [
        'Political philosophy', 'Marxism', 'Scientific socialism', 'Historical materialism', 'Dialectical materialism',
        'Class struggle', 'Karl Marx', 'Friedrich Engels', 'The Communist Manifesto', 'Das Kapital', 'Socialism',
        'Communism', 'Anarchism', 'Liberalism', 'Classical liberalism', 'Conservatism', 'Nationalism',
        'Social contract', 'Thomas More', 'Niccolo Machiavelli', 'Alexis de Tocqueville', 'Hannah Arendt',
        'Antonio Gramsci', 'Frankfurt School', 'Critical theory', 'Post-structuralism', 'Deconstruction',
        'Luddite', 'Neo-Luddism', 'Technological unemployment', 'Industrial Revolution',
    ],
    'economic_ideologies': [
        'Neoliberalism', 'Laissez-faire', 'Economic liberalism', 'Free market', 'Market economy', 'Capitalism',
        'Invisible hand', 'Adam Smith', 'The Wealth of Nations', 'Austrian school of economics', 'Friedrich Hayek',
        'Milton Friedman', 'Monetarism', 'Deregulation', 'Privatization', 'Keynesian economics',
        'John Maynard Keynes', 'Political economy', 'Economic determinism', 'Welfare capitalism',
    ],
    'psychology_psyche': [
        'Psychology', 'Mind', 'Psyche (psychology)', 'Sigmund Freud', 'Psychoanalysis', 'Carl Jung',
        'Analytical psychology', 'Collective unconscious', 'Id, ego and superego', 'Defense mechanism',
        'Behaviorism', 'B. F. Skinner', 'Classical conditioning', 'Operant conditioning', 'Ivan Pavlov',
        'Cognitive psychology', 'Cognitive science', 'Jean Piaget', 'Developmental psychology',
        'Social psychology', 'Abnormal psychology', 'Personality psychology', 'Humanistic psychology',
        'Abraham Maslow', 'Maslow\'s hierarchy of needs', 'Carl Rogers', 'Emotion', 'Motivation',
        'Memory', 'Learning', 'Perception', 'Consciousness', 'Unconscious mind', 'Cognitive bias',
    ],
    'gender_identity_critical_theory': [
        'Judith Butler', 'Gender performativity', 'Gender studies', 'Queer theory', 'Feminist theory',
        'Feminist philosophy', 'Intersectionality', 'Kimberle Crenshaw', 'Gender identity', 'Postmodernism',
        'Postmodern philosophy', 'Julia Kristeva', 'Luce Irigaray', 'Helene Cixous', 'Donna Haraway',
        'The Second Sex', 'Simone de Beauvoir', 'Michel Foucault', 'Discipline and Punish',
        'The History of Sexuality', 'Jacques Derrida', 'Of Grammatology', 'Differance',
    ],
    'ethics_society_csr': [
        'Corporate social responsibility', 'Business ethics', 'Stakeholder theory', 'Corporate governance',
        'Environmental, social, and governance', 'Triple bottom line', 'Social responsibility',
        'Sustainability', 'Applied ethics', 'Environmental ethics', 'Bioethics', 'Normative ethics',
        'Consequentialism', 'Moral relativism', 'Moral universalism', 'Human rights', 'Social justice',
    ],
}

MOCK_QUERY_TITLES = [
    'Corporate social responsibility', 'Scientific socialism', 'Neoliberalism', 'Laissez-faire',
    'George Berkeley', 'Subjective idealism', 'Immaterialism', 'Marxism', 'Jacques Derrida',
    'Judith Butler', 'Luddite', 'Business ethics', 'Stakeholder theory', 'Historical materialism',
]

print('target article total:', sum(SUBJECT_QUOTAS.values()))
display(pd.DataFrame([{'subject': k, 'quota': v, 'category_roots': len(CATEGORY_SPECS[k]), 'manual_seeds': len(MANUAL_SEED_TITLES[k])} for k, v in SUBJECT_QUOTAS.items()]))

## 2. Discovery Helpers

The builder uses Wikipedia category members for page discovery, then resolves each page title to a Wikidata QID through `pageprops.wikibase_item`. This keeps the final DragonLLM fetch QID-based like the existing builders.

In [ ]:
QID_RE = re.compile(r'Q\d+')
SPACE_RE = re.compile(r'\s+')

EXCLUDE_TITLE_RE = re.compile(
    r'\b(list of|lists of|index of|outline of|timeline of|bibliography of|glossary of|portal:|template:|category:|file:)\b',
    re.I,
)

EXCLUDE_SUBCATEGORY_RE = re.compile(
    r'\b(by country|by nationality|by language|by ethnicity|by religion|births|deaths|alumni|journals|magazines|'
    r'periodicals|organizations|societies|associations|awards|conferences|festivals|publishers|university|universities|'
    r'wikiproject|templates|redirects|stubs|lists)\b',
    re.I,
)

LOW_VALUE_TITLE_RE = re.compile(
    r'\b(journal|magazine|newsletter|conference|award|prize|university press)\b',
    re.I,
)

def normalize_text(value):
    if value is None:
        return ''
    return SPACE_RE.sub(' ', str(value).replace('\u00a0', ' ')).strip()

def clean_qid(value):
    if isinstance(value, dict):
        for key in ['id', 'qid', 'value', 'entity']:
            if key in value:
                found = clean_qid(value[key])
                if found:
                    return found
    if isinstance(value, (list, tuple)):
        for item in value:
            found = clean_qid(item)
            if found:
                return found
    match = QID_RE.search(str(value or ''))
    return match.group(0) if match else ''

def normalize_title(value):
    title = normalize_text(value).replace('_', ' ')
    return title.lower()

def canonical_url(value):
    text = normalize_text(value)
    if not text:
        return ''
    return text.replace('http://', 'https://').split('#', 1)[0].rstrip('/')

def safe_int(value, default=0):
    try:
        return int(float(value))
    except Exception:
        return default

def safe_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return default

def load_json(path, default):
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return default

def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

def is_allowed_page_title(title):
    title = normalize_text(title)
    if not title or EXCLUDE_TITLE_RE.search(title):
        return False
    if title.endswith('(disambiguation)'):
        return False
    if LOW_VALUE_TITLE_RE.search(title):
        return False
    return True

def is_allowed_subcategory(title):
    title = normalize_text(title).replace('Category:', '')
    if not title or EXCLUDE_SUBCATEGORY_RE.search(title):
        return False
    if EXCLUDE_TITLE_RE.search(title):
        return False
    return True

def wikipedia_api(params, timeout=60, max_retries=6, base_sleep=2.0):
    params = dict(params)
    params.setdefault('format', 'json')
    params.setdefault('formatversion', '2')
    url = 'https://en.wikipedia.org/w/api.php?' + urllib.parse.urlencode(params)
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'})
    last_exc = None
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.loads(response.read().decode('utf-8'))
        except urllib.error.HTTPError as exc:
            last_exc = exc
            if exc.code not in (429, 500, 502, 503, 504):
                raise
            retry_after = exc.headers.get('Retry-After')
            wait = int(retry_after) if retry_after and retry_after.isdigit() else min(90, base_sleep * (2 ** attempt))
            print(f'HTTP {exc.code}. Waiting {wait:.1f}s before retry {attempt + 1}/{max_retries}...')
            time.sleep(wait)
        except urllib.error.URLError as exc:
            last_exc = exc
            wait = min(90, base_sleep * (2 ** attempt))
            print(f'URL error {exc}. Waiting {wait:.1f}s before retry {attempt + 1}/{max_retries}...')
            time.sleep(wait)
    raise RuntimeError(f'Wikipedia API failed after {max_retries} retries: {last_exc}')

category_cache = load_json(DISCOVERY_CACHE_JSON, {})
resolution_cache = load_json(RESOLUTION_CACHE_JSON, {})

In [ ]:
def category_title(name):
    name = normalize_text(name).replace('_', ' ')
    return name if name.startswith('Category:') else f'Category:{name}'

def fetch_category_members(category, cmtype='page|subcat'):
    category = category_title(category)
    cache_key = f'category::{category}::{cmtype}'
    if cache_key in category_cache:
        return category_cache[cache_key]

    members = []
    params = {
        'action': 'query',
        'list': 'categorymembers',
        'cmtitle': category,
        'cmtype': cmtype,
        'cmlimit': 'max',
    }
    while True:
        data = wikipedia_api(params)
        batch = data.get('query', {}).get('categorymembers', [])
        members.extend(batch)
        cont = data.get('continue', {})
        if 'cmcontinue' not in cont:
            break
        params['cmcontinue'] = cont['cmcontinue']

    category_cache[cache_key] = members
    save_json(DISCOVERY_CACHE_JSON, category_cache)
    return members

def discover_subject_pages(subject, specs, quota):
    target_candidates = max(quota * CATEGORY_CANDIDATE_MULTIPLIER, quota + 100)
    rows = []
    seen_titles = set()

    for spec in specs:
        root_category = category_title(spec['category'])
        max_depth = int(spec.get('max_depth', MAX_CATEGORY_DEPTH_DEFAULT))
        root_weight = int(spec.get('weight', 50))
        queue = [(root_category, 0, root_category)]
        seen_categories = set()

        while queue and len(rows) < target_candidates:
            current_category, depth, path = queue.pop(0)
            if current_category in seen_categories:
                continue
            seen_categories.add(current_category)

            try:
                members = fetch_category_members(current_category)
            except Exception as exc:
                print(f'skipping category {current_category}: {exc}')
                continue

            for member in members:
                ns = int(member.get('ns', -1))
                title = normalize_text(member.get('title', ''))

                if ns == 0:
                    if not is_allowed_page_title(title):
                        continue
                    key = normalize_title(title)
                    if key in seen_titles:
                        continue
                    seen_titles.add(key)
                    rows.append({
                        'title': title,
                        'subject': subject,
                        'source_channel': 'wikipedia_category',
                        'source_query': root_category,
                        'category_path': path,
                        'category_depth': depth,
                        'priority_score': root_weight + max(0, 50 - depth * 15),
                    })
                    if len(rows) >= target_candidates:
                        break

                elif ns == 14 and depth < max_depth:
                    if not is_allowed_subcategory(title):
                        continue
                    queue.append((title, depth + 1, f'{path} > {title}'))

    return rows

def manual_seed_rows():
    rows = []
    for subject, titles in MANUAL_SEED_TITLES.items():
        for title in titles:
            if is_allowed_page_title(title):
                rows.append({
                    'title': title,
                    'subject': subject,
                    'source_channel': 'manual_seed_title',
                    'source_query': 'manual_seed_titles',
                    'category_path': '',
                    'category_depth': 0,
                    'priority_score': 10_000,
                })
    for title in MOCK_QUERY_TITLES:
        rows.append({
            'title': title,
            'subject': 'question_driven_seed',
            'source_channel': 'mock_question_seed',
            'source_query': 'mock_questions_from_planning',
            'category_path': '',
            'category_depth': 0,
            'priority_score': 20_000,
        })
    return rows

def chunked(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]

def resolve_titles_to_pages(titles, batch_size=45):
    out = {}
    missing = []
    unique_titles = []
    seen = set()
    for title in titles:
        key = normalize_title(title)
        if key and key not in seen:
            seen.add(key)
            unique_titles.append(title)

    to_fetch = []
    for title in unique_titles:
        cache_key = f'title::{title}'
        if cache_key in resolution_cache:
            out[title] = resolution_cache[cache_key]
        else:
            to_fetch.append(title)

    for batch in tqdm(list(chunked(to_fetch, batch_size)), desc='resolving titles to QIDs'):
        data = wikipedia_api({
            'action': 'query',
            'redirects': '1',
            'prop': 'pageprops|info',
            'ppprop': 'wikibase_item',
            'inprop': 'url',
            'titles': '|'.join(batch),
        })
        query = data.get('query', {})
        pages = query.get('pages', [])
        pages_by_norm = {}
        alias_to_target = {}
        for item in query.get('normalized', []):
            alias_to_target[normalize_title(item.get('from', ''))] = normalize_title(item.get('to', ''))
        for item in query.get('redirects', []):
            alias_to_target[normalize_title(item.get('from', ''))] = normalize_title(item.get('to', ''))
        for page in pages:
            requested_title = page.get('title', '')
            pages_by_norm[normalize_title(requested_title)] = page

        for title in batch:
            lookup_key = normalize_title(title)
            page = pages_by_norm.get(lookup_key)
            for _ in range(3):
                if page is not None:
                    break
                next_key = alias_to_target.get(lookup_key)
                if not next_key or next_key == lookup_key:
                    break
                lookup_key = next_key
                page = pages_by_norm.get(lookup_key)
            payload = None
            if page and not page.get('missing'):
                qid = clean_qid(page.get('pageprops', {}).get('wikibase_item', ''))
                if qid:
                    payload = {
                        'qid': qid,
                        'label': normalize_text(page.get('title', title)),
                        'wikipedia_url': canonical_url(page.get('fullurl', '')),
                        'wikidata_description': normalize_text(page.get('description', '')),
                        'pageid': page.get('pageid', ''),
                    }
            if payload is None:
                missing.append(title)
                payload = {}
            resolution_cache[f'title::{title}'] = payload
            out[title] = payload

        save_json(RESOLUTION_CACHE_JSON, resolution_cache)

    return out, missing

def split_pipe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    return [x.strip() for x in re.split(r'\s*\|\s*', str(value)) if x.strip()]

def join_unique(values):
    out = []
    for value in values:
        for part in split_pipe(value):
            if part and part not in out:
                out.append(part)
    return ' | '.join(out)

def choose_primary_subject(subjects, channels):
    subject_set = set(split_pipe(subjects))
    channel_set = set(split_pipe(channels))
    if 'question_driven_seed' in subject_set or 'mock_question_seed' in channel_set:
        for subject in SUBJECT_PRIORITY:
            if any(seed.lower() in subjects.lower() for seed in [subject]):
                return subject
    for subject in SUBJECT_PRIORITY:
        if subject in subject_set:
            return subject
    return sorted(subject_set)[0] if subject_set else 'core_philosophy'

def infer_subject_from_label(label):
    blob = str(label or '').lower()
    if any(term in blob for term in ['corporate social responsibility', 'business ethics', 'stakeholder', 'governance', 'social responsibility', 'ethics']):
        return 'ethics_society_csr'
    if any(term in blob for term in ['neoliberal', 'laissez', 'free market', 'capitalism', 'economic liberalism', 'monetarism']):
        return 'economic_ideologies'
    if any(term in blob for term in ['marx', 'socialism', 'luddite', 'political', 'class struggle', 'critical theory']):
        return 'political_social_thought'
    if any(term in blob for term in ['butler', 'gender', 'queer', 'feminist', 'intersectionality']):
        return 'gender_identity_critical_theory'
    if any(term in blob for term in ['freud', 'jung', 'psychology', 'psychoanalysis', 'behaviorism', 'cognitive']):
        return 'psychology_psyche'
    return 'core_philosophy'

def infer_page_type(row):
    blob = ' | '.join(str(row.get(col, '')) for col in ['label', 'subjects', 'source_channels', 'category_paths']).lower()
    title = str(row.get('label', '')).lower()
    if any(term in blob for term in ['philosophers', 'psychologists', 'psychoanalysts']) or any(name in title for name in ['marx', 'freud', 'kant', 'derrida', 'butler', 'berkeley']):
        return 'thinker'
    if any(term in blob for term in ['ideologies', 'liberalism', 'marxism', 'socialism', 'capitalism', 'laissez-faire', 'neoliberalism']):
        return 'ideology'
    if any(term in blob for term in ['theories', 'theory']):
        return 'theory'
    if any(term in blob for term in ['concepts', 'concept']):
        return 'concept'
    if any(term in blob for term in ['schools and traditions', 'school', 'tradition']):
        return 'school_or_tradition'
    if any(term in blob for term in ['business ethics', 'corporate social responsibility', 'applied ethics']):
        return 'applied_ethics'
    return 'article'

## 3. Run Wikipedia Discovery And Select QIDs

Outputs:

```text
logs/philosophy_psychology_wiki_v1_candidate_qids.csv
logs/philosophy_psychology_wiki_v1_selected_qids.csv
```

Inspect the selected CSV after the first run. Manual and mock-question seeds receive the highest priority.

In [ ]:
if not RUN_WIKIPEDIA_DISCOVERY and SELECTED_QIDS_CSV.exists():
    candidates = pd.read_csv(CANDIDATE_QIDS_CSV).fillna('')
    selected = pd.read_csv(SELECTED_QIDS_CSV).fillna('')
    print('loaded existing candidates:', len(candidates))
    print('loaded existing selected:', len(selected))
else:
    page_rows = manual_seed_rows()

    print('manual/mock seed title rows:', len(page_rows))
    for subject, specs in CATEGORY_SPECS.items():
        rows = discover_subject_pages(subject, specs, SUBJECT_QUOTAS[subject])
        page_rows.extend(rows)
        print(f'{subject}: category page rows {len(rows)}')

    page_df = pd.DataFrame(page_rows).drop_duplicates(subset=['title', 'subject', 'source_channel', 'source_query']).fillna('')
    print('page discovery rows:', len(page_df))

    resolved_by_title, missing_titles = resolve_titles_to_pages(page_df['title'].tolist())
    if missing_titles:
        missing_path = LOGS_DIR / f'{RUN_LABEL}_unresolved_titles.csv'
        pd.DataFrame({'title': missing_titles}).drop_duplicates().to_csv(missing_path, index=False)
        print('unresolved title count:', len(set(missing_titles)), '| saved:', missing_path)

    candidate_rows = []
    for _, row in page_df.iterrows():
        resolved = resolved_by_title.get(row['title'], {})
        qid = clean_qid(resolved.get('qid', ''))
        if not qid:
            continue
        label = resolved.get('label') or row['title']
        candidate_rows.append({
            'qid': qid,
            'label': label,
            'wikipedia_url': resolved.get('wikipedia_url', ''),
            'wikidata_description': resolved.get('wikidata_description', ''),
            'pageid': resolved.get('pageid', ''),
            'subject': row['subject'],
            'source_channel': row['source_channel'],
            'source_query': row['source_query'],
            'category_path': row.get('category_path', ''),
            'category_depth': safe_int(row.get('category_depth', 0)),
            'priority_score': safe_float(row.get('priority_score', 0)),
        })

    if not MANUAL_SEED_QIDS_CSV.exists():
        pd.DataFrame(columns=['qid', 'label', 'wikipedia_url', 'subject', 'notes']).to_csv(MANUAL_SEED_QIDS_CSV, index=False)
        print('created optional manual QID seed file:', MANUAL_SEED_QIDS_CSV)
    else:
        manual_qids = pd.read_csv(MANUAL_SEED_QIDS_CSV).fillna('')
        for _, row in manual_qids.iterrows():
            qid = clean_qid(row.get('qid', ''))
            if qid:
                candidate_rows.append({
                    'qid': qid,
                    'label': normalize_text(row.get('label', '')),
                    'wikipedia_url': canonical_url(row.get('wikipedia_url', '')),
                    'wikidata_description': '',
                    'pageid': '',
                    'subject': normalize_text(row.get('subject', 'question_driven_seed')) or 'question_driven_seed',
                    'source_channel': 'manual_seed_qid_csv',
                    'source_query': str(MANUAL_SEED_QIDS_CSV.name),
                    'category_path': '',
                    'category_depth': 0,
                    'priority_score': 30_000,
                })

    raw_candidates = pd.DataFrame(candidate_rows).fillna('')
    if raw_candidates.empty:
        raise RuntimeError('No QID candidates were produced. Check category names and network access.')

    grouped_rows = []
    for qid, group in raw_candidates.groupby('qid', dropna=False):
        subjects = join_unique(group['subject'].tolist())
        channels = join_unique(group['source_channel'].tolist())
        category_paths = join_unique(group['category_path'].tolist())
        source_queries = join_unique(group['source_query'].tolist())
        best = group.sort_values('priority_score', ascending=False).iloc[0]
        primary_subject = choose_primary_subject(subjects, channels)
        if primary_subject not in SUBJECT_QUOTAS:
            primary_subject = infer_subject_from_label(best.get('label', ''))
        grouped_rows.append({
            'qid': qid,
            'label': best.get('label', ''),
            'wikipedia_url': best.get('wikipedia_url', ''),
            'wikidata_description': best.get('wikidata_description', ''),
            'pageid': best.get('pageid', ''),
            'subjects': subjects,
            'primary_subject': primary_subject,
            'source_channels': channels,
            'source_queries': source_queries,
            'category_paths': category_paths,
            'max_priority_score': float(pd.to_numeric(group['priority_score'], errors='coerce').fillna(0).max()),
            'min_category_depth': int(pd.to_numeric(group['category_depth'], errors='coerce').fillna(99).min()),
        })

    candidates = pd.DataFrame(grouped_rows).fillna('')
    candidates['page_type'] = candidates.apply(infer_page_type, axis=1)
    candidates['selection_score'] = candidates['max_priority_score'] + candidates['page_type'].map({
        'thinker': 350,
        'concept': 300,
        'theory': 280,
        'ideology': 260,
        'school_or_tradition': 240,
        'applied_ethics': 220,
        'article': 0,
    }).fillna(0) - candidates['min_category_depth'].astype(float) * 10

    candidates = candidates.sort_values(['primary_subject', 'selection_score', 'label'], ascending=[True, False, True])
    candidates.to_csv(CANDIDATE_QIDS_CSV, index=False)

    selected_parts = []
    for subject in SUBJECT_PRIORITY:
        subject_df = candidates[candidates['primary_subject'] == subject].copy()
        quota = SUBJECT_QUOTAS[subject]
        selected_parts.append(subject_df.head(quota))

    selected = pd.concat(selected_parts, ignore_index=True).drop_duplicates(subset=['qid'])

    # Force high-priority mock/manual pages even if a quota is already full.
    force_mask = candidates['source_channels'].str.contains('mock_question_seed|manual_seed_title|manual_seed_qid_csv', regex=True, na=False)
    selected = pd.concat([selected, candidates[force_mask]], ignore_index=True).drop_duplicates(subset=['qid'])
    selected = selected.sort_values(['primary_subject', 'selection_score', 'label'], ascending=[True, False, True]).reset_index(drop=True)
    selected['selection_status'] = 'selected'
    selected['taxonomy_source'] = TAXONOMY_SOURCE
    selected['taxonomy_confidence'] = selected['source_channels'].apply(lambda x: 0.95 if 'manual_seed' in str(x) or 'mock_question_seed' in str(x) else 0.82)
    selected.to_csv(SELECTED_QIDS_CSV, index=False)

print('candidate qids:', len(candidates))
print('selected qids:', len(selected))
print('saved candidates:', CANDIDATE_QIDS_CSV)
print('saved selected:', SELECTED_QIDS_CSV)
display(selected['primary_subject'].value_counts().rename_axis('primary_subject').reset_index(name='selected'))
display(selected[['qid', 'label', 'primary_subject', 'page_type', 'selection_score', 'source_channels', 'wikipedia_url']].head(80))

In [ ]:
import pandas as pd

selected = pd.read_csv(SELECTED_QIDS_CSV).fillna("")

def infer_subject_from_label(label):
    blob = str(label or "").lower()

    if any(t in blob for t in [
        "corporate social responsibility", "business ethics", "stakeholder",
        "corporate governance", "social responsibility",
        "environmental, social, and governance", "applied ethics"
    ]):
        return "ethics_society_csr"

    if any(t in blob for t in [
        "neoliberal", "laissez", "free market", "capitalism",
        "economic liberalism", "monetarism", "privatization", "deregulation"
    ]):
        return "economic_ideologies"

    if any(t in blob for t in [
        "marx", "scientific socialism", "historical materialism",
        "socialism", "luddite", "class struggle", "critical theory"
    ]):
        return "political_social_thought"

    if any(t in blob for t in [
        "butler", "gender", "queer", "feminist", "intersectionality"
    ]):
        return "gender_identity_critical_theory"

    if any(t in blob for t in [
        "freud", "jung", "psychology", "psychoanalysis",
        "behaviorism", "cognitive"
    ]):
        return "psychology_psyche"

    return None

mask = selected["source_channels"].astype(str).str.contains(
    "manual_seed_title|mock_question_seed|manual_seed_qid_csv",
    regex=True,
    na=False,
)

selected.loc[mask, "primary_subject"] = selected.loc[mask, "label"].apply(
    lambda x: infer_subject_from_label(x) or "core_philosophy"
)

selected.to_csv(SELECTED_QIDS_CSV, index=False)

print("patched selected:", len(selected))
display(selected[["qid", "label", "primary_subject", "page_type", "source_channels"]].head(80))

## 4. Audit Coverage For Mock Questions

This quick check makes sure the pages implied by the planning examples are selected before spending time on the DragonLLM fetch.

In [ ]:
selected = pd.read_csv(SELECTED_QIDS_CSV).fillna('')
mock_norm = {normalize_title(t): t for t in MOCK_QUERY_TITLES}
selected_norm = {normalize_title(t): t for t in selected['label'].tolist()}
coverage_rows = []
for norm, expected_title in mock_norm.items():
    matched = norm in selected_norm
    close = [title for key, title in selected_norm.items() if norm in key or key in norm][:5]
    coverage_rows.append({
        'expected_title': expected_title,
        'selected_exact_label': selected_norm.get(norm, ''),
        'exact_match': matched,
        'close_selected_labels': ' | '.join(close),
    })
coverage = pd.DataFrame(coverage_rows)
display(coverage)
if not coverage['exact_match'].all():
    print('Some mock titles did not resolve as exact selected labels. Check redirects/aliases and add QIDs to:', MANUAL_SEED_QIDS_CSV)

In [ ]:
import pandas as pd

selected = pd.read_csv(SELECTED_QIDS_CSV).fillna("")

required = {
    "Corporate social responsibility": ["Corporate social responsibility"],
    "Scientific socialism": ["Scientific socialism"],
    "Neoliberalism": ["Neoliberalism"],
    "Laissez-faire": ["Laissez-faire"],
    "George Berkeley": ["George Berkeley"],
    "Berkeley material objects": ["George Berkeley", "Subjective idealism"],
    "Immaterialism coverage": ["Immaterialism", "Subjective idealism", "George Berkeley"],
    "Marxism": ["Marxism"],
    "Jacques Derrida": ["Jacques Derrida"],
    "Judith Butler": ["Judith Butler"],
    "Luddism": ["Luddite", "Neo-Luddism"],
    "Business ethics": ["Business ethics"],
    "Stakeholder theory": ["Stakeholder theory"],
    "Historical materialism": ["Historical materialism"],
}

rows = []
labels = selected["label"].astype(str).tolist()

for check_name, acceptable_labels in required.items():
    hits = selected[selected["label"].isin(acceptable_labels)].copy()
    rows.append({
        "check": check_name,
        "ok": len(hits) > 0,
        "matched_labels": " | ".join(hits["label"].tolist()),
        "matched_qids": " | ".join(hits["qid"].tolist()),
    })

audit = pd.DataFrame(rows)
display(audit)

print("selected rows:", len(selected))
print("unique qids:", selected["qid"].nunique())
print("missing qid:", int((selected["qid"].astype(str).str.len() == 0).sum()))
print("missing url:", int((selected["wikipedia_url"].astype(str).str.len() == 0).sum()))
print("duplicate qids:", int(selected["qid"].duplicated().sum()))

display(selected["primary_subject"].value_counts().rename_axis("primary_subject").reset_index(name="selected"))

if audit["ok"].all() and selected["qid"].nunique() == len(selected):
    print("GO: puoi procedere al fetch DragonLLM.")
else:
    print("CHECK: guarda le righe False o eventuali duplicati prima del fetch.")

## 5. Fetch Articles From DragonLLM And Save Dataset

If the DragonLLM dataset is gated, authenticate before running this section:

```python
from huggingface_hub import login
login()
```

In [ ]:
# Optional if Hugging Face access is not already configured:
from huggingface_hub import login
login()

In [ ]:
def row_value(row, *keys, default=''):
    for key in keys:
        if isinstance(row, dict) and key in row:
            value = row.get(key)
            if value is not None and not (isinstance(value, str) and value == ''):
                return value
    return default

def jsonish(value):
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return normalize_text(value)

def extract_row_qid(row):
    for key in ['qid', 'wikidata_qid', 'entity', 'wikidata_id']:
        qid = clean_qid(row.get(key, '')) if isinstance(row, dict) else ''
        if qid:
            return qid
    pageprops = row.get('pageprops') if isinstance(row, dict) else None
    if isinstance(pageprops, dict):
        return clean_qid(pageprops.get('wikibase_item', ''))
    return ''

def make_article_row(source_row, meta):
    qid = extract_row_qid(source_row) or clean_qid(meta.get('qid', ''))
    title = normalize_text(row_value(source_row, 'title', default=meta.get('label', '')))
    url = canonical_url(row_value(source_row, 'url', 'canonicalurl', default=meta.get('wikipedia_url', ''))) or canonical_url(meta.get('wikipedia_url', ''))
    text = normalize_text(row_value(source_row, 'text', default=''))
    doc_id = f'wiki_{qid}' if qid else f"wiki_{normalize_title(title).replace(' ', '_')}"
    return {
        'doc_id': doc_id,
        'text': text,
        'title': title,
        'url': url,
        'qid': qid,
        'source_dataset': SOURCE_WIKI_DATASET,
        'source_type': WIKI_SOURCE_TYPE,
        'license': WIKIPEDIA_LICENSE,
        'subject': meta.get('primary_subject', 'core_philosophy'),
        'topic': meta.get('label', title),
        'page_type': meta.get('page_type', 'article'),
        'taxonomy_source': meta.get('taxonomy_source', TAXONOMY_SOURCE),
        'taxonomy_confidence': safe_float(meta.get('taxonomy_confidence', 0.82), 0.82),
        'wikidata_label': meta.get('label', title),
        'wikidata_description': meta.get('wikidata_description', ''),
        'wikipedia_url': meta.get('wikipedia_url', url),
        'source_channels': meta.get('source_channels', ''),
        'source_queries': meta.get('source_queries', ''),
        'category_paths': meta.get('category_paths', ''),
        'selection_score': safe_float(meta.get('selection_score', 0.0)),
        'categories': jsonish(row_value(source_row, 'categories', default='')),
        'infobox': jsonish(row_value(source_row, 'infobox', default='')),
        'token_count': safe_int(row_value(source_row, 'token_count', default=0)),
        'revision_id': normalize_text(row_value(source_row, 'revision_id', 'revid', default='')),
        'revdate': normalize_text(row_value(source_row, 'revdate', 'timestamp', default='')),
    }

selected = pd.read_csv(SELECTED_QIDS_CSV).fillna('')
selected['qid'] = selected['qid'].map(clean_qid)
selected = selected[selected['qid'] != ''].copy()
selected_by_qid = {row['qid']: row.to_dict() for _, row in selected.iterrows()}
target_qids = set(selected_by_qid)

if not RUN_FETCH_AND_BUILD_DATASET:
    print('Skipping DragonLLM fetch because RUN_FETCH_AND_BUILD_DATASET=False')
else:
    if OUTPUT_DATASET_DIR.exists() and not OVERWRITE_OUTPUT_DATASET:
        raise FileExistsError(f'{OUTPUT_DATASET_DIR} already exists. Set OVERWRITE_OUTPUT_DATASET=True only if you want to regenerate it.')

    print('target qids:', len(target_qids))
    source_stream = load_dataset(SOURCE_WIKI_DATASET, split='train', streaming=True)

    found_by_qid = {}
    scanned = 0
    for row in source_stream:
        scanned += 1
        qid = extract_row_qid(row)
        if qid in target_qids and qid not in found_by_qid:
            found_by_qid[qid] = dict(row)
            meta = selected_by_qid[qid]
            print(f"found {len(found_by_qid):>4}/{len(target_qids)}: {qid} | {meta.get('label', row.get('title', ''))}")
            if len(found_by_qid) == len(target_qids):
                break
        if SOURCE_SCAN_PROGRESS_EVERY and scanned % SOURCE_SCAN_PROGRESS_EVERY == 0:
            print(f'scanned {scanned:,} rows; found {len(found_by_qid):,}/{len(target_qids):,}')
        if MAX_SOURCE_ROWS_TO_SCAN is not None and scanned >= MAX_SOURCE_ROWS_TO_SCAN:
            print(f'stopped after MAX_SOURCE_ROWS_TO_SCAN={MAX_SOURCE_ROWS_TO_SCAN}')
            break

    added_rows = []
    skipped_rows = []
    for qid, source_row in found_by_qid.items():
        meta = selected_by_qid[qid]
        article = make_article_row(source_row, meta)
        if len(article['text']) < MIN_ARTICLE_CHARS:
            skipped_rows.append({**meta, 'status': 'skipped_short_text', 'text_chars': len(article['text'])})
            continue
        if not is_allowed_page_title(article['title']):
            skipped_rows.append({**meta, 'status': 'skipped_title_filter', 'text_chars': len(article['text'])})
            continue
        added_rows.append(article)

    missing_qids = sorted(target_qids - set(found_by_qid))
    fetch_report = []
    for qid, meta in selected_by_qid.items():
        status = 'added' if any(row['qid'] == qid for row in added_rows) else 'missing_from_dragonllm' if qid in missing_qids else 'skipped_quality_filter'
        fetch_report.append({
            'qid': qid,
            'label': meta.get('label', ''),
            'primary_subject': meta.get('primary_subject', ''),
            'status': status,
            'wikipedia_url': meta.get('wikipedia_url', ''),
        })

    if not added_rows:
        raise RuntimeError('No article rows were added. Check DragonLLM access, QID fields and filters.')

    if OUTPUT_DATASET_DIR.exists() and OVERWRITE_OUTPUT_DATASET:
        shutil.rmtree(OUTPUT_DATASET_DIR)

    ds = DatasetDict({'train': Dataset.from_list(added_rows)})
    ds.save_to_disk(str(OUTPUT_DATASET_DIR))

    pd.DataFrame(fetch_report).to_csv(FETCH_REPORT_CSV, index=False)
    pd.DataFrame(added_rows).drop(columns=['text'], errors='ignore').to_csv(ADDED_ARTICLES_CSV, index=False)
    pd.DataFrame(skipped_rows).to_csv(SKIPPED_ARTICLES_CSV, index=False)

    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'run_label': RUN_LABEL,
        'source_wiki_dataset': SOURCE_WIKI_DATASET,
        'output_dataset_dir': str(OUTPUT_DATASET_DIR),
        'selection_strategy': 'curated Wikipedia categories + manual/mock seeds -> QIDs -> DragonLLM QID filtering',
        'candidate_qids': int(len(pd.read_csv(CANDIDATE_QIDS_CSV))) if CANDIDATE_QIDS_CSV.exists() else None,
        'selected_qids': int(len(selected)),
        'source_rows_scanned': int(scanned),
        'found_in_dragonllm': int(len(found_by_qid)),
        'articles_added': int(len(added_rows)),
        'articles_skipped': int(len(skipped_rows)),
        'missing_qids': int(len(missing_qids)),
        'subject_counts': pd.DataFrame(added_rows)['subject'].value_counts().to_dict(),
        'page_type_counts': pd.DataFrame(added_rows)['page_type'].value_counts().to_dict(),
        'min_article_chars': MIN_ARTICLE_CHARS,
    }
    BUILD_REPORT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

    print('source rows scanned:', scanned)
    print('articles added:', len(added_rows))
    print('missing qids:', len(missing_qids))
    print('saved dataset:', OUTPUT_DATASET_DIR)
    print('saved report:', BUILD_REPORT_JSON)
    print(json.dumps(report, indent=2, ensure_ascii=False))

## 6. Dataset Audit

In [ ]:
if OUTPUT_DATASET_DIR.exists():
    ds = load_from_disk(str(OUTPUT_DATASET_DIR))['train']
    df = ds.to_pandas().fillna('')
    print(ds)
    print('rows:', len(df))
    print('unique qids:', df['qid'].map(clean_qid).replace('', pd.NA).dropna().nunique())
    display(df['subject'].value_counts().rename_axis('subject').reset_index(name='rows'))
    display(df['page_type'].value_counts().rename_axis('page_type').reset_index(name='rows'))
    display(df[['qid', 'title', 'subject', 'page_type', 'topic', 'token_count', 'url']].head(80))
else:
    print('Dataset does not exist yet:', OUTPUT_DATASET_DIR)

In [ ]:
!ollama --version

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

with open("/tmp/ollama_philosophy_index.log", "ab") as log_file:
    ollama_process = subprocess.Popen(
        ["bash", "-lc", "OLLAMA_CONTEXT_LENGTH=8192 ollama serve"],
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(8)

In [ ]:
EMBEDDING_MODEL = "hf.co/unsloth/embeddinggemma-300m-GGUF:BF16"

!ollama pull {EMBEDDING_MODEL}
!ollama list

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama",
)

resp = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="George Berkeley and subjective idealism"
)

print(len(resp.data[0].embedding))

## 7. Ollama Embedding Preflight

The index build uses the same local Ollama-compatible embedding endpoint used by the existing builders.

In [ ]:
EMBEDDING_MODEL = 'hf.co/unsloth/embeddinggemma-300m-GGUF:BF16'
EMBEDDING_BASE_URL = 'http://localhost:11434/v1/'
EMBEDDING_API_KEY = 'ollama'

START_OLLAMA_SERVER = False
PULL_EMBEDDING_MODEL = False

if START_OLLAMA_SERVER:
    import subprocess
    subprocess.run(['killall', 'ollama'], check=False)
    with open('/tmp/ollama_philosophy_psychology_index.log', 'ab') as log_file:
        subprocess.Popen(['bash', '-lc', 'OLLAMA_CONTEXT_LENGTH=8192 ollama serve'], stdout=log_file, stderr=log_file)
    time.sleep(5)

if PULL_EMBEDDING_MODEL:
    import subprocess
    subprocess.run(['ollama', 'pull', EMBEDDING_MODEL], check=True)

def http_json(url, timeout=8, max_retries=2):
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'})
    last_exc = None
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.loads(response.read().decode('utf-8'))
        except Exception as exc:
            last_exc = exc
            time.sleep(1 + attempt)
    raise RuntimeError(f'Could not fetch {url}: {last_exc}')

def assert_embedding_endpoint_ready():
    tags_url = EMBEDDING_BASE_URL.rstrip('/')
    if tags_url.endswith('/v1'):
        tags_url = tags_url[:-3]
    tags_url = tags_url.rstrip('/') + '/api/tags'
    payload = http_json(tags_url, timeout=5)
    print('embedding endpoint reachable:', tags_url)
    print(json.dumps(payload, indent=2, ensure_ascii=False)[:1200])

if RUN_INDEX_BUILD:
    assert_embedding_endpoint_ready()
else:
    print('Skipping endpoint preflight because RUN_INDEX_BUILD=False')

## 8. Build FAISS Index

Creates:

```text
Indexes/philosophy_psychology_wiki_v1/index.faiss
Indexes/philosophy_psychology_wiki_v1/index.pkl
```

In [ ]:
if not RUN_INDEX_BUILD:
    print('Skipping index build because RUN_INDEX_BUILD=False')
else:
    from langchain_core.documents import Document
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_text_splitters import TokenTextSplitter

    if not OUTPUT_DATASET_DIR.exists():
        raise FileNotFoundError(f'Build dataset first: {OUTPUT_DATASET_DIR}')
    if OUTPUT_INDEX_DIR.exists():
        if not OVERWRITE_INDEX:
            raise FileExistsError(f'{OUTPUT_INDEX_DIR} exists. Set OVERWRITE_INDEX=True to replace it.')
        shutil.rmtree(OUTPUT_INDEX_DIR)

    assert_embedding_endpoint_ready()

    ds = load_from_disk(str(OUTPUT_DATASET_DIR))['train']
    df = ds.to_pandas().fillna('')

    splitter = TokenTextSplitter(encoding_name='cl100k_base', chunk_size=512, chunk_overlap=128)
    metadata_cols = [
        'doc_id', 'title', 'url', 'qid', 'source_dataset', 'source_type', 'license',
        'subject', 'topic', 'page_type', 'taxonomy_source', 'taxonomy_confidence',
        'wikidata_label', 'wikidata_description', 'wikipedia_url', 'source_channels',
        'source_queries', 'category_paths', 'selection_score', 'token_count', 'revdate',
    ]

    docs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='chunking wiki articles'):
        text = str(row.get('text', '') or '')
        if not text.strip():
            continue
        base_meta = {col: row.get(col, '') for col in metadata_cols if col in df.columns}
        base_meta['taxonomy_confidence'] = safe_float(base_meta.get('taxonomy_confidence', 0.0), 0.0)
        base_meta['selection_score'] = safe_float(base_meta.get('selection_score', 0.0), 0.0)
        chunks = splitter.split_text(text)
        for chunk_idx, chunk in enumerate(chunks):
            meta = dict(base_meta)
            meta['chunk_id'] = f"{base_meta.get('doc_id', '')}_chunk_{chunk_idx:04d}"
            meta['chunk_index'] = chunk_idx
            docs.append(Document(page_content=chunk, metadata=meta))

    if not docs:
        raise RuntimeError('No chunks produced for indexing.')

    print('chunks to embed:', len(docs))
    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=EMBEDDING_BASE_URL,
        api_key=EMBEDDING_API_KEY,
        check_embedding_ctx_length=False,
    )

    vectorstore = None
    batch_size = 64
    for start in tqdm(range(0, len(docs), batch_size), desc='embedding wiki chunks'):
        batch = docs[start:start + batch_size]
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)

    vectorstore.save_local(str(OUTPUT_INDEX_DIR))
    index_report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'input_dataset_dir': str(OUTPUT_DATASET_DIR),
        'output_index_dir': str(OUTPUT_INDEX_DIR),
        'article_rows': int(len(df)),
        'chunks': int(len(docs)),
        'chunk_size': 512,
        'chunk_overlap': 128,
        'embedding_model': EMBEDDING_MODEL,
        'metadata_cols': metadata_cols + ['chunk_id', 'chunk_index'],
        'subject_counts': df['subject'].value_counts().to_dict(),
        'page_type_counts': df['page_type'].value_counts().to_dict(),
        'index_files': sorted(p.name for p in OUTPUT_INDEX_DIR.iterdir()),
    }
    INDEX_REPORT_JSON.write_text(json.dumps(index_report, indent=2, ensure_ascii=False), encoding='utf-8')
    print('saved index:', OUTPUT_INDEX_DIR)
    print(json.dumps(index_report, indent=2, ensure_ascii=False))

## 9. Smoke Test

These queries mirror the expected category shape: definitional knowledge, thinkers, political/economic thought, psychology and theory.

In [ ]:
if OUTPUT_INDEX_DIR.exists():
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS

    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=EMBEDDING_BASE_URL,
        api_key=EMBEDDING_API_KEY,
        check_embedding_ctx_length=False,
    )
    store = FAISS.load_local(str(OUTPUT_INDEX_DIR), embeddings, allow_dangerous_deserialization=True)

    queries = [
        'Which best describes corporate social responsibility?',
        'What is the fundamental principle of scientific socialism?',
        'What is neoliberalism?',
        'What does laissez-faire economics argue?',
        'What did George Berkeley believe about material objects?',
        'What is deconstruction in Jacques Derrida?',
        'What is Judith Butler known for?',
        'What is Luddism?',
        'What is psychoanalysis?',
    ]

    rows = []
    for query in queries:
        for rank, (doc, score) in enumerate(store.similarity_search_with_score(query, k=5), start=1):
            rows.append({
                'query': query,
                'rank': rank,
                'score': float(score),
                'title': doc.metadata.get('title', ''),
                'subject': doc.metadata.get('subject', ''),
                'page_type': doc.metadata.get('page_type', ''),
                'topic': doc.metadata.get('topic', ''),
                'qid': doc.metadata.get('qid', ''),
                'text_preview': doc.page_content[:350],
            })
    display(pd.DataFrame(rows))
else:
    print('Index does not exist yet:', OUTPUT_INDEX_DIR)

In [ ]:
import json
import pickle
import pandas as pd
from pathlib import Path
from IPython.display import display

print("Dataset:", OUTPUT_DATASET_DIR)
print("Index:", OUTPUT_INDEX_DIR)

# Dataset article distribution
ds = load_from_disk(str(OUTPUT_DATASET_DIR))["train"]
df = ds.to_pandas().fillna("")

print("\nDATASET")
print("articles:", len(df))
print("unique qids:", df["qid"].nunique())
print("total tokens:", int(pd.to_numeric(df["token_count"], errors="coerce").fillna(0).sum()))

display(df["subject"].value_counts().rename_axis("subject").reset_index(name="articles"))
display(df["page_type"].value_counts().rename_axis("page_type").reset_index(name="articles"))

display(
    df.groupby(["subject", "page_type"])
      .size()
      .reset_index(name="articles")
      .sort_values(["subject", "articles"], ascending=[True, False])
)

# Index chunk distribution
class FakeInMemoryDocstore:
    def __setstate__(self, state):
        self.__dict__.update(state)

class FakeDocument:
    def __setstate__(self, state):
        self.__dict__.update(state)

class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "langchain_community.docstore.in_memory" and name == "InMemoryDocstore":
            return FakeInMemoryDocstore
        if module == "langchain_core.documents.base" and name == "Document":
            return FakeDocument
        return super().find_class(module, name)

pkl_path = Path(OUTPUT_INDEX_DIR) / "index.pkl"

with pkl_path.open("rb") as f:
    docstore, index_to_docstore_id = SafeUnpickler(f).load()

docs = list(docstore._dict.values())

rows = []
for doc in docs:
    payload = getattr(doc, "__dict__", {})
    if "metadata" not in payload and "__dict__" in payload:
        payload = payload["__dict__"]

    meta = payload.get("metadata", {})
    rows.append({
        "chunk_id": meta.get("chunk_id", ""),
        "qid": meta.get("qid", ""),
        "title": meta.get("title", ""),
        "subject": meta.get("subject", ""),
        "page_type": meta.get("page_type", ""),
        "chunk_index": meta.get("chunk_index", ""),
    })

chunks = pd.DataFrame(rows).fillna("")

print("\nINDEX")
print("chunks:", len(chunks))
print("unique article qids in index:", chunks["qid"].nunique())
print("index mapping size:", len(index_to_docstore_id))

display(chunks["subject"].value_counts().rename_axis("subject").reset_index(name="chunks"))
display(chunks["page_type"].value_counts().rename_axis("page_type").reset_index(name="chunks"))

display(
    chunks.groupby(["subject", "page_type"])
          .size()
          .reset_index(name="chunks")
          .sort_values(["subject", "chunks"], ascending=[True, False])
)

display(
    chunks.groupby(["qid", "title", "subject", "page_type"])
          .size()
          .reset_index(name="chunks")
          .sort_values("chunks", ascending=False)
          .head(30)
)